# Day 6：手写 RAG 系统实践 — 从切分、检索到增强生成

> **目标**：从零写出一个完整的 RAG 管线，覆盖文档切分（含 overlap）、多种向量化方案（词袋 / TF-IDF）、BM25 关键词检索、稠密向量检索、混合检索、重排、增强生成与检索质量评估；所有代码零外部依赖，突出系统数据流。

---

## 总体流程

```text
Part 1: 准备知识库
Part 2: 文档切分（固定长度 / overlap / 结构化）
Part 3: Embedding 方案对比（词袋 vs TF-IDF）
Part 4: BM25 关键词检索
Part 5: 稠密向量检索
Part 6: 混合检索
Part 7: 简单重排器
Part 8: RAG Pipeline（检索 + 重排 + 生成）
Part 9: 检索质量评估
Part 10: 升级方向与面试要点
```

In [1]:
from __future__ import annotations

from typing import List, Dict, Any, Set, Tuple
from collections import Counter
from dataclasses import dataclass, field
import math
import json
import re

## Part 1：准备知识库

用课程内容模拟一个多文档知识库。真实系统里这些文本可能来自 Markdown、PDF、网页或数据库。

In [2]:
documents = [
    {
        "doc_id": "week2_transformer",
        "text": (
            "Transformer 是一种基于自注意力机制的序列到序列模型。"
            "它由 Encoder 和 Decoder 两部分组成。"
            "自注意力通过 Query、Key、Value 三个矩阵计算注意力权重。"
            "多头注意力让模型在不同子空间捕捉不同模式。"
            "位置编码用正弦函数或可学习向量表示 token 位置。"
        ),
    },
    {
        "doc_id": "week3_gpt",
        "text": (
            "GPT 是一种基于 Decoder-only Transformer 的自回归语言模型。"
            "它通过 next token prediction 任务进行预训练。"
            "GPT-3 证明了 scaling law：模型越大、数据越多，性能越好。"
            "In-context learning 让模型可以通过 few-shot 示例完成任务。"
        ),
    },
    {
        "doc_id": "week6_lora",
        "text": (
            "LoRA 是一种参数高效微调方法。"
            "它假设权重更新矩阵是低秩的，因此用两个小矩阵 BA 近似 DeltaW。"
            "LoRA 只训练 A 和 B 两个低秩矩阵，冻结预训练权重。"
            "推理时可以把 BA 合并回原权重，实现零额外开销。"
            "通常对 Attention 的 Wq 和 Wv 注入 LoRA 效果最好。"
        ),
    },
    {
        "doc_id": "week6_qlora",
        "text": (
            "QLoRA 在 LoRA 基础上引入 NF4 量化。"
            "NF4 是一种 4-bit 量化方式，专为正态分布权重设计。"
            "双重量化进一步压缩量化常数本身的存储开销。"
            "分页优化器用 CPU 内存缓解 GPU 显存不足。"
            "QLoRA 让在单张消费级 GPU 上微调 65B 模型成为可能。"
        ),
    },
    {
        "doc_id": "week8_agent",
        "text": (
            "Agent 是 LLM 加工具调用加状态管理的系统组合。"
            "ReAct 让模型在 Thought、Action、Observation 之间循环。"
            "Function Calling 用 JSON Schema 描述工具接口。"
            "Agent 需要 Memory 管理短期和长期上下文。"
            "工具注册表统一管理所有可调用工具。"
        ),
    },
    {
        "doc_id": "week8_rag",
        "text": (
            "RAG 是检索增强生成，让模型能访问外部知识库。"
            "离线阶段完成文档切分、向量化和索引构建。"
            "在线阶段完成 query 向量化、top-k 召回和增强生成。"
            "Naive RAG 容易出现召回不足、精度不够和上下文稀释。"
            "优化方向包括 query 改写、混合检索、重排和压缩。"
        ),
    },
    {
        "doc_id": "week8_embedding",
        "text": (
            "文本嵌入模型把语义相近的句子映射到相近向量。"
            "常见 pooling 方式有 CLS pooling 和 Mean pooling。"
            "双塔架构把 query 和 doc 分别编码，适合大规模检索。"
            "交叉编码器精度更高但速度慢，适合重排。"
            "InfoNCE 是对比学习的核心损失函数。"
        ),
    },
]

print(f"知识库共 {len(documents)} 篇文档:")
for doc in documents:
    print(f"  {doc['doc_id']}: {len(doc['text'])} 字符")

知识库共 7 篇文档:
  week2_transformer: 142 字符
  week3_gpt: 159 字符
  week6_lora: 145 字符
  week6_qlora: 135 字符
  week8_agent: 153 字符
  week8_rag: 132 字符
  week8_embedding: 135 字符


## Part 2：文档切分

切分质量直接影响检索效果。这里实现三种切分策略并对比。

### 2.1 固定长度切分

In [3]:
def fixed_split(text: str, chunk_size: int = 60) -> List[str]:
    """固定长度切分，不考虑语义边界。"""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size
    return chunks


demo_text = documents[0]["text"]
print("固定长度切分 (chunk_size=60):")
for i, chunk in enumerate(fixed_split(demo_text, 60)):
    print(f"  [{i}] ({len(chunk)}字) {chunk}")

固定长度切分 (chunk_size=60):
  [0] (60字) Transformer 是一种基于自注意力机制的序列到序列模型。它由 Encoder 和 Decoder 两部分组成。自
  [1] (60字) 注意力通过 Query、Key、Value 三个矩阵计算注意力权重。多头注意力让模型在不同子空间捕捉不同模式。位置编码用
  [2] (22字) 正弦函数或可学习向量表示 token 位置。


### 2.2 Overlap 切分

滑窗 + overlap 能缓解语义边界被切断的问题。

In [4]:
def overlap_split(text: str, chunk_size: int = 60, overlap: int = 15) -> List[str]:
    """带 overlap 的滑窗切分。"""
    if chunk_size <= overlap:
        raise ValueError("chunk_size 必须大于 overlap")
    chunks = []
    start = 0
    step = chunk_size - overlap
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += step
    return chunks


print("Overlap 切分 (chunk_size=60, overlap=15):")
for i, chunk in enumerate(overlap_split(demo_text, 60, 15)):
    print(f"  [{i}] ({len(chunk)}字) {chunk}")

Overlap 切分 (chunk_size=60, overlap=15):
  [0] (60字) Transformer 是一种基于自注意力机制的序列到序列模型。它由 Encoder 和 Decoder 两部分组成。自
  [1] (60字) Decoder 两部分组成。自注意力通过 Query、Key、Value 三个矩阵计算注意力权重。多头注意力让模型在不同
  [2] (52字) 力权重。多头注意力让模型在不同子空间捕捉不同模式。位置编码用正弦函数或可学习向量表示 token 位置。


### 2.3 句子级切分

按句号分割后合并，更尊重语义边界。

In [5]:
def sentence_split(text: str, max_chars: int = 80) -> List[str]:
    """按句子分割后合并到不超过 max_chars 的 chunk。"""
    sentences = [s.strip() for s in re.split(r"[。！？]", text) if s.strip()]
    chunks = []
    current = ""
    for sent in sentences:
        candidate = current + sent + "。" if current else sent + "。"
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = sent + "。"
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


print("句子级切分 (max_chars=80):")
for i, chunk in enumerate(sentence_split(demo_text, 80)):
    print(f"  [{i}] ({len(chunk)}字) {chunk}")

句子级切分 (max_chars=80):
  [0] (59字) Transformer 是一种基于自注意力机制的序列到序列模型。它由 Encoder 和 Decoder 两部分组成。
  [1] (56字) 自注意力通过 Query、Key、Value 三个矩阵计算注意力权重。多头注意力让模型在不同子空间捕捉不同模式。
  [2] (27字) 位置编码用正弦函数或可学习向量表示 token 位置。


In [6]:
# 用 overlap_split 构建全部 chunk
ALL_CHUNKS = []
for doc in documents:
    for idx, chunk_text in enumerate(overlap_split(doc["text"], chunk_size=60, overlap=15)):
        ALL_CHUNKS.append({
            "chunk_id": f"{doc['doc_id']}_{idx}",
            "doc_id": doc["doc_id"],
            "text": chunk_text,
        })

print(f"共生成 {len(ALL_CHUNKS)} 个 chunk")
for c in ALL_CHUNKS[:5]:
    print(f"  {c['chunk_id']}: {c['text'][:50]}...")

共生成 23 个 chunk
  week2_transformer_0: Transformer 是一种基于自注意力机制的序列到序列模型。它由 Encoder 和 Decod...
  week2_transformer_1: Decoder 两部分组成。自注意力通过 Query、Key、Value 三个矩阵计算注意力权重。多...
  week2_transformer_2: 力权重。多头注意力让模型在不同子空间捕捉不同模式。位置编码用正弦函数或可学习向量表示 token 位...
  week3_gpt_0: GPT 是一种基于 Decoder-only Transformer 的自回归语言模型。它通过 ne...
  week3_gpt_1: 通过 next token prediction 任务进行预训练。GPT-3 证明了 scaling...


## Part 3：Embedding 方案对比

为了让 notebook 零依赖可运行，这里实现两种 embedding：词袋和 TF-IDF。TF-IDF 通过 IDF 加权应该比纯词袋更有区分力。

### 3.1 词袋 Embedding

In [7]:
VOCAB = [
    "transformer", "注意力", "自注意力", "encoder", "decoder",
    "gpt", "自回归", "预训练", "scaling",
    "lora", "低秩", "微调", "冻结", "合并",
    "qlora", "量化", "nf4", "显存",
    "agent", "工具", "react", "memory",
    "rag", "检索", "知识库", "召回", "重排",
    "embedding", "向量", "双塔", "infonce", "对比学习",
    "query", "chunk", "索引",
]


def bag_of_words_embed(text: str) -> List[float]:
    """词袋 embedding：统计 VOCAB 中每个词在文本中的出现次数。"""
    text_lower = text.lower()
    return [float(text_lower.count(token.lower())) for token in VOCAB]


vec = bag_of_words_embed("LoRA 是一种低秩微调方法")
nonzero = [(VOCAB[i], v) for i, v in enumerate(vec) if v > 0]
print("词袋示例:", nonzero)

词袋示例: [('lora', 1.0), ('低秩', 1.0), ('微调', 1.0)]


### 3.2 TF-IDF Embedding

In [8]:
class TFIDFEmbedder:
    """
    基于 VOCAB 的 TF-IDF embedding。

    TF(t,d) = count(t in d) / len(d)
    IDF(t) = log(N / (1 + df(t)))
    """

    def __init__(self, vocab: List[str]):
        self.vocab = vocab
        self.idf: List[float] = []
        self.fitted = False

    def fit(self, corpus: List[str]):
        """在语料上计算 IDF。"""
        n = len(corpus)
        df = [0.0] * len(self.vocab)
        for doc in corpus:
            doc_lower = doc.lower()
            for i, token in enumerate(self.vocab):
                if token.lower() in doc_lower:
                    df[i] += 1
        self.idf = [math.log(n / (1 + d)) for d in df]
        self.fitted = True

    def embed(self, text: str) -> List[float]:
        """返回 TF-IDF 向量。"""
        if not self.fitted:
            raise RuntimeError("请先调用 fit()")
        text_lower = text.lower()
        text_len = max(len(text_lower), 1)
        vec = []
        for i, token in enumerate(self.vocab):
            tf = text_lower.count(token.lower()) / text_len
            vec.append(tf * self.idf[i])
        return vec


tfidf = TFIDFEmbedder(VOCAB)
corpus = [c["text"] for c in ALL_CHUNKS]
tfidf.fit(corpus)

vec_tfidf = tfidf.embed("LoRA 是一种低秩微调方法")
nonzero_tfidf = [(VOCAB[i], round(v, 4)) for i, v in enumerate(vec_tfidf) if v > 0]
print("TF-IDF 示例:", nonzero_tfidf)

TF-IDF 示例: [('lora', 0.096), ('低秩', 0.1455), ('微调', 0.1455)]


### 3.3 余弦相似度

In [9]:
def cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
    """计算两个向量的余弦相似度。"""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


# 快速对比词袋 vs TF-IDF 的区分能力
query = "LoRA 为什么参数高效"
q_bow = bag_of_words_embed(query)
q_tfidf = tfidf.embed(query)

print(f"Query: {query}\n")
print(f"{'Chunk':<30} {'词袋 sim':>10} {'TF-IDF sim':>12}")
print("-" * 55)
for c in ALL_CHUNKS[:8]:
    bow_sim = cosine_similarity(q_bow, bag_of_words_embed(c["text"]))
    tfidf_sim = cosine_similarity(q_tfidf, tfidf.embed(c["text"]))
    label = c["chunk_id"][:25]
    print(f"{label:<30} {bow_sim:>10.4f} {tfidf_sim:>12.4f}")

Query: LoRA 为什么参数高效

Chunk                              词袋 sim   TF-IDF sim
-------------------------------------------------------
week2_transformer_0                0.0000       0.0000
week2_transformer_1                0.0000       0.0000
week2_transformer_2                0.0000       0.0000
week3_gpt_0                        0.0000       0.0000
week3_gpt_1                        0.0000       0.0000
week3_gpt_2                        0.0000       0.0000
week3_gpt_3                        0.0000       0.0000
week6_lora_0                       0.8165       0.6822


## Part 4：BM25 关键词检索

BM25 是经典的关键词检索算法，在精确匹配场景下仍然很强。纯 Python 实现。

In [10]:
def tokenize_chinese(text: str) -> List[str]:
    """简易中文分词：按字 + 英文按词。"""
    tokens = []
    buf = ""
    for ch in text:
        if ch.isascii() and ch.isalnum():
            buf += ch
        else:
            if buf:
                tokens.append(buf.lower())
                buf = ""
            if ch.strip() and not ch.isascii():
                tokens.append(ch)
    if buf:
        tokens.append(buf.lower())
    return tokens


class BM25Retriever:
    """
    BM25 检索器。

    score(q, d) = sum_{t in q} IDF(t) * (tf * (k1+1)) / (tf + k1*(1 - b + b*dl/avgdl))
    """

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.chunks: List[Dict] = []
        self.doc_tokens: List[List[str]] = []
        self.avgdl: float = 0
        self.idf: Dict[str, float] = {}

    def fit(self, chunks: List[Dict]):
        """索引所有 chunk。"""
        self.chunks = chunks
        self.doc_tokens = [tokenize_chinese(c["text"]) for c in chunks]
        n = len(chunks)
        self.avgdl = sum(len(t) for t in self.doc_tokens) / max(n, 1)

        # 计算 IDF
        df: Dict[str, int] = {}
        for tokens in self.doc_tokens:
            for t in set(tokens):
                df[t] = df.get(t, 0) + 1
        self.idf = {
            t: math.log((n - d + 0.5) / (d + 0.5) + 1)
            for t, d in df.items()
        }

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """对 query 做 BM25 检索。"""
        q_tokens = tokenize_chinese(query)
        scores = []
        for idx, doc_toks in enumerate(self.doc_tokens):
            score = 0.0
            dl = len(doc_toks)
            tf_map = Counter(doc_toks)
            for qt in q_tokens:
                if qt not in self.idf:
                    continue
                tf = tf_map.get(qt, 0)
                idf = self.idf[qt]
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                score += idf * numerator / denominator
            scores.append((score, idx))
        scores.sort(reverse=True)
        return [
            {
                "score": round(s, 4),
                "chunk_id": self.chunks[i]["chunk_id"],
                "doc_id": self.chunks[i]["doc_id"],
                "text": self.chunks[i]["text"],
            }
            for s, i in scores[:top_k]
            if s > 0
        ]


bm25 = BM25Retriever()
bm25.fit(ALL_CHUNKS)

print("BM25 检索: 'LoRA 低秩微调'")
for item in bm25.search("LoRA 低秩微调", top_k=5):
    print(f"  [{item['score']:.4f}] {item['chunk_id']}: {item['text'][:50]}...")

BM25 检索: 'LoRA 低秩微调'
  [9.6242] week6_lora_0: LoRA 是一种参数高效微调方法。它假设权重更新矩阵是低秩的，因此用两个小矩阵 BA 近似 Delt...
  [5.5629] week6_lora_1:  DeltaW。LoRA 只训练 A 和 B 两个低秩矩阵，冻结预训练权重。推理时可以把 BA 合并...
  [4.0718] week6_qlora_2: 缓解 GPU 显存不足。QLoRA 让在单张消费级 GPU 上微调 65B 模型成为可能。...
  [1.8741] week8_agent_3: 。工具注册表统一管理所有可调用工具。...
  [1.7262] week6_lora_2: BA 合并回原权重，实现零额外开销。通常对 Attention 的 Wq 和 Wv 注入 LoRA ...


## Part 5：稠密向量检索

使用 TF-IDF embedding + 余弦相似度做稠密检索。

In [ ]:
class DenseRetriever:
    """稠密向量检索器。"""

    def __init__(self, embed_fn):
        self.embed_fn = embed_fn
        self.chunks: List[Dict] = []
        self.vectors: List[List[float]] = []

    def fit(self, chunks: List[Dict]):
        """构建索引：对所有 chunk 做向量化。"""
        self.chunks = chunks
        self.vectors = [self.embed_fn(c["text"]) for c in chunks]

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """检索最相似的 top-k chunk。"""
        q_vec = self.embed_fn(query)
        scored = []
        for idx, d_vec in enumerate(self.vectors):
            sim = cosine_similarity(q_vec, d_vec)
            scored.append((sim, idx))
        scored.sort(reverse=True)
        return [
            {
                "score": round(s, 4),
                "chunk_id": self.chunks[i]["chunk_id"],
                "doc_id": self.chunks[i]["doc_id"],
                "text": self.chunks[i]["text"],
            }
            for s, i in scored[:top_k]
            if s > 0
        ]


dense = DenseRetriever(tfidf.embed)
dense.fit(ALL_CHUNKS)

print("稠密检索 (TF-IDF): 'RAG 如何检索外部知识库'")
for item in dense.search("RAG 如何检索外部知识库", top_k=5):
    print(f"  [{item['score']:.4f}] {item['chunk_id']}: {item['text'][:50]}...")

## Part 6：混合检索

将 BM25 和 Dense 的检索结果做分数融合。混合检索能同时利用精确匹配和语义理解。

In [ ]:
class HybridRetriever:
    """
    混合检索器：融合 BM25 和 Dense 检索结果。

    final_score = alpha * norm(dense_score) + (1-alpha) * norm(bm25_score)
    """

    def __init__(self, dense_retriever: DenseRetriever, bm25_retriever: BM25Retriever, alpha: float = 0.5):
        self.dense = dense_retriever
        self.bm25 = bm25_retriever
        self.alpha = alpha

    def _normalize_scores(self, results: List[Dict]) -> Dict[str, float]:
        """Min-max 归一化。"""
        if not results:
            return {}
        scores = [r["score"] for r in results]
        min_s, max_s = min(scores), max(scores)
        rng = max_s - min_s if max_s > min_s else 1.0
        return {r["chunk_id"]: (r["score"] - min_s) / rng for r in results}

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """混合检索。"""
        dense_results = self.dense.search(query, top_k=top_k * 2)
        bm25_results = self.bm25.search(query, top_k=top_k * 2)

        dense_scores = self._normalize_scores(dense_results)
        bm25_scores = self._normalize_scores(bm25_results)

        all_ids = set(dense_scores.keys()) | set(bm25_scores.keys())

        chunk_map = {}
        for r in dense_results + bm25_results:
            chunk_map[r["chunk_id"]] = r

        combined = []
        for cid in all_ids:
            ds = dense_scores.get(cid, 0.0)
            bs = bm25_scores.get(cid, 0.0)
            final = self.alpha * ds + (1 - self.alpha) * bs
            combined.append((final, cid))

        combined.sort(reverse=True)
        return [
            {
                "score": round(s, 4),
                "chunk_id": cid,
                "doc_id": chunk_map[cid]["doc_id"],
                "text": chunk_map[cid]["text"],
            }
            for s, cid in combined[:top_k]
        ]


hybrid = HybridRetriever(dense, bm25, alpha=0.5)

print("混合检索: 'LoRA 为什么参数高效'")
for item in hybrid.search("LoRA 为什么参数高效", top_k=5):
    print(f"  [{item['score']:.4f}] {item['chunk_id']}: {item['text'][:50]}...")

## Part 7：简单重排器

先用快方法（BM25 / Dense）粗召回 top-20，再用更精细的方法重排 top-5。这里用 query-chunk 的词重叠度模拟交叉编码器的重排效果。

In [ ]:
class SimpleReranker:
    """
    基于 query-chunk 词重叠的简单重排器。

    真实系统中这里会使用交叉编码器（Cross-Encoder）。
    """

    def rerank(self, query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
        q_tokens = set(tokenize_chinese(query))
        scored = []
        for idx, c in enumerate(candidates):
            c_tokens = set(tokenize_chinese(c["text"]))
            overlap = len(q_tokens & c_tokens)
            coverage = overlap / max(len(q_tokens), 1)
            scored.append((coverage, idx, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [
            {**c, "rerank_score": round(s, 4)}
            for s, _, c in scored[:top_k]
        ]


reranker = SimpleReranker()

# 先粗召回再重排
candidates = hybrid.search("QLoRA 为什么能省显存", top_k=8)
reranked = reranker.rerank("QLoRA 为什么能省显存", candidates, top_k=3)

print("重排前 (top-5):")
for item in candidates[:5]:
    print(f"  [{item['score']:.4f}] {item['chunk_id']}: {item['text'][:45]}...")

print("\n重排后 (top-3):")
for item in reranked:
    print(f"  [rerank={item['rerank_score']:.4f}] {item['chunk_id']}: {item['text'][:45]}...")

## Part 8：RAG Pipeline

把检索 + 重排 + 生成组装成完整的 RAG 管线。

In [ ]:
def build_rag_prompt(query: str, retrieved_chunks: List[Dict]) -> str:
    """构造 RAG prompt。"""
    context = "\n\n".join(
        f"[{item['doc_id']} | chunk={item.get('chunk_id', '?')}] {item['text']}"
        for item in retrieved_chunks
    )
    return f"""你是课程助教，请基于给定资料回答问题。

资料:
{context}

问题:
{query}

要求:
1. 优先基于资料作答
2. 若资料不足，要明确说明
3. 引用资料来源"""


def simple_generator(query: str, retrieved_chunks: List[Dict]) -> str:
    """
    模拟 LLM 生成。

    真实系统中这里会调用大语言模型。
    """
    if not retrieved_chunks:
        return "当前知识库中没有足够证据回答这个问题。"

    best = retrieved_chunks[0]
    if best.get("rerank_score", best.get("score", 0)) == 0:
        return "检索到的资料与问题相关度不高，无法给出可靠答案。"

    sources = set(item["doc_id"] for item in retrieved_chunks[:3])
    merged = " ".join(item["text"] for item in retrieved_chunks[:3])
    source_str = ", ".join(sources)
    return f"基于检索到的资料（来源: {source_str}），可以回答：{merged}"


class RAGPipeline:
    """完整 RAG 管线：检索 -> 重排 -> 生成。"""

    def __init__(self, retriever, reranker=None, generator_fn=None):
        self.retriever = retriever
        self.reranker = reranker
        self.generator_fn = generator_fn or simple_generator

    def run(self, query: str, retrieve_k: int = 8, final_k: int = 3) -> Dict[str, Any]:
        retrieved = self.retriever.search(query, top_k=retrieve_k)

        if self.reranker:
            reranked = self.reranker.rerank(query, retrieved, top_k=final_k)
        else:
            reranked = retrieved[:final_k]

        prompt = build_rag_prompt(query, reranked)
        answer = self.generator_fn(query, reranked)

        return {
            "query": query,
            "retrieved": retrieved[:5],
            "reranked": reranked,
            "prompt": prompt,
            "answer": answer,
        }


rag = RAGPipeline(hybrid, reranker)

In [ ]:
test_queries = [
    "LoRA 为什么参数高效？",
    "RAG 的完整流程是什么？",
    "QLoRA 如何降低显存？",
    "Transformer 的自注意力机制是什么？",
    "Agent 和 RAG 有什么关系？",
]

for query in test_queries:
    print("=" * 70)
    result = rag.run(query)
    print(f"Query: {result['query']}")
    print(f"Reranked chunks:")
    for item in result["reranked"]:
        score_key = "rerank_score" if "rerank_score" in item else "score"
        print(f"  [{item[score_key]:.4f}] {item['chunk_id']}")
    print(f"Answer: {result['answer'][:100]}...")
    print()

## Part 9：检索质量评估

RAG 系统不能只看"能不能回答"，必须量化检索质量。这里实现三个核心指标：

- **Recall@k**：top-k 中包含相关文档的比例
- **Precision@k**：top-k 中相关文档的占比
- **MRR**：第一个相关文档的排名倒数

In [ ]:
def recall_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:
    """top-k 中包含了多少个相关文档。"""
    if not relevant_ids:
        return 0.0
    hits = sum(1 for rid in retrieved_ids[:k] if rid in relevant_ids)
    return hits / len(relevant_ids)


def precision_at_k(retrieved_ids: List[str], relevant_ids: Set[str], k: int) -> float:
    """top-k 中有多大比例是相关文档。"""
    if k == 0:
        return 0.0
    hits = sum(1 for rid in retrieved_ids[:k] if rid in relevant_ids)
    return hits / k


def mrr(retrieved_ids: List[str], relevant_ids: Set[str]) -> float:
    """第一个相关文档排在第几位（倒数）。"""
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

In [ ]:
# 构造评测集：(query, 相关的 doc_id 集合)
eval_set = [
    ("LoRA 的核心原理是什么", {"week6_lora"}),
    ("QLoRA 如何节省显存", {"week6_qlora"}),
    ("RAG 的检索流程", {"week8_rag"}),
    ("Transformer 自注意力机制", {"week2_transformer"}),
    ("Agent 如何调用工具", {"week8_agent"}),
    ("文本嵌入和对比学习", {"week8_embedding"}),
    ("GPT 的预训练方式", {"week3_gpt"}),
]

# 对比三种检索方案
retrievers = {
    "BM25": bm25,
    "Dense (TF-IDF)": dense,
    "Hybrid": hybrid,
}

print(f"{'Retriever':<20} {'Recall@3':>10} {'Recall@5':>10} {'P@3':>10} {'MRR':>10}")
print("-" * 65)

for name, ret in retrievers.items():
    r3_list, r5_list, p3_list, mrr_list = [], [], [], []
    for query, relevant_docs in eval_set:
        results = ret.search(query, top_k=10)
        retrieved_doc_ids = [r["doc_id"] for r in results]

        r3_list.append(recall_at_k(retrieved_doc_ids, relevant_docs, 3))
        r5_list.append(recall_at_k(retrieved_doc_ids, relevant_docs, 5))
        p3_list.append(precision_at_k(retrieved_doc_ids, relevant_docs, 3))
        mrr_list.append(mrr(retrieved_doc_ids, relevant_docs))

    avg_r3 = sum(r3_list) / len(r3_list)
    avg_r5 = sum(r5_list) / len(r5_list)
    avg_p3 = sum(p3_list) / len(p3_list)
    avg_mrr = sum(mrr_list) / len(mrr_list)
    print(f"{name:<20} {avg_r3:>10.4f} {avg_r5:>10.4f} {avg_p3:>10.4f} {avg_mrr:>10.4f}")

## Part 10：升级方向与面试要点

### 10.1 如何升级为真实 RAG

| 当前实现 | 真实系统 |
|----------|----------|
| VOCAB 词袋 / TF-IDF | Sentence-Transformers / BGE / Jina Embeddings |
| 暴力遍历检索 | FAISS / Chroma / Milvus |
| 固定长度切分 | token-based chunking + overlap + 结构化切分 |
| 规则化 generator | 真实 LLM (GPT-4 / Qwen / LLaMA) |
| 简单词重叠重排 | Cross-Encoder reranker (bge-reranker) |
| 无 query 改写 | HyDE / Query Expansion |

### 10.2 面试高频手写

1. **BM25 公式与实现**：IDF + TF 归一化 + 文档长度惩罚
2. **余弦相似度**：`dot(a,b) / (norm(a) * norm(b))`
3. **RAG Pipeline**：检索 -> 重排 -> prompt 构造 -> 生成
4. **Recall@k / MRR**：检索质量的基本评估指标
5. **混合检索**：BM25 + Dense 分数融合

### 10.3 练习题

1. 给 `overlap_split` 增加 `min_chunk_size` 参数，丢弃过短的末尾 chunk。
2. 给 `DenseRetriever` 增加 `min_score` 阈值过滤。
3. 实现一个简单的 query 改写函数（如关键词扩展）。
4. 在 RAG prompt 中加入"引用来源"的格式要求，并验证 generator 是否遵循。
5. 把这个 RAG 检索器封装为 Day3 Agent 的一个工具。
6. 用真实 embedding 模型（如 `sentence-transformers`）替换 TF-IDF。